# AGCN — Baseline oficial do Qwen3-4B atual

GPU CUDA + **Qwen3-4B-Q4_K_M.gguf** + llama.cpp b11264. Usa PresenterBrain, prompt e validators atuais, sem fine-tuning. As 25 situações congeladas nunca entram no treino.


## 0. Ative a GPU

No Colab: **Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU**.

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception:
    raise RuntimeError('GPU não está ativa. Ative T4 GPU e rode novamente.')


## 1. Clonar a branch de treinamento

O repositório está público: não é necessário fornecer senha nem token. A main não é alterada.


In [ ]:
import os, pathlib, subprocess
repo_dir = pathlib.Path('/content/AGCN-Live-Voice')
branch = 'research/agcn-presenter-training-v1'
if not repo_dir.exists():
    subprocess.run(['git','clone','--depth','1','--branch',branch,'https://github.com/adrse/AGCN-Live-Voice.git',str(repo_dir)], check=True)
os.chdir(repo_dir)
assert subprocess.check_output(['git','branch','--show-current'], text=True).strip() == branch
print('Branch:', branch)
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())


## 2. Preparar llama.cpp CUDA e o GGUF oficial

Usa binários oficiais com SHA256 conferido; compila a mesma versão se o binário for incompatível. Não substitui o GGUF por Transformers.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','requests','huggingface_hub'], check=True)
p = subprocess.Popen([sys.executable,'-u','training/scripts/setup_colab_baseline.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
assert p.wait() == 0, 'Preparação falhou; verifique os logs acima.'


## 3. Conferir as 25 situações congeladas

Elas não entram no treino.

In [ ]:
import json, pathlib, hashlib
frozen = pathlib.Path('training/evaluation/frozen_eval_v0.1.jsonl')
rows = [json.loads(x) for x in frozen.read_text(encoding='utf-8').splitlines() if x.strip()]
assert len(rows) == 25 and len({x['id'] for x in rows}) == 25
frozen_sha256 = hashlib.sha256(frozen.read_bytes()).hexdigest()
print('Situações congeladas:', len(rows), 'SHA256:', frozen_sha256)


## 4. Rodar o baseline oficial com CUDA

O runner preserva o comportamento de produção e verifica o offload CUDA antes dos casos. Falhas dos validators são registradas como comportamento do antes.


In [ ]:
import os, subprocess, sys
os.environ.update(json.loads(pathlib.Path('training/baselines/colab_runtime_env.json').read_text()))
os.environ['AGCN_BASELINE_REQUIRE_CUDA'] = '1'
os.environ['LOCALAPPDATA'] = '/content/agcn-baseline-logs'
os.environ['LLAMA_ARG_LOG_VERBOSITY'] = '4'
print(pathlib.Path('training/baselines/qwen3_4b_current_v0.1_runtime.json').read_text(), flush=True)
cmd = [sys.executable,'-u','training/scripts/run_baseline_current_qwen.py','training/evaluation/frozen_eval_v0.1.jsonl','training/baselines/qwen3_4b_current_v0.1.jsonl','training/baselines/qwen3_4b_current_v0.1_summary.json']
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='', flush=True)
assert p.wait() == 0, 'Baseline falhou; verifique os logs acima.'


## 5. Ver o resumo

In [ ]:
summary_path = pathlib.Path('training/baselines/qwen3_4b_current_v0.1_summary.json')
summary = json.loads(summary_path.read_text())
results = [json.loads(x) for x in pathlib.Path('training/baselines/qwen3_4b_current_v0.1.jsonl').read_text().splitlines() if x.strip()]
assert len(results) == 25 and {r['id'] for r in results} == {r['id'] for r in rows}
assert hashlib.sha256(frozen.read_bytes()).hexdigest() == frozen_sha256
print(json.dumps(summary, indent=2, ensure_ascii=False))


## 6. Salvar no Google Drive

In [ ]:
from google.colab import drive
import shutil
# A autorização de conta é feita manualmente pelo titular.
drive.mount('/content/drive')
dest = pathlib.Path('/content/drive/MyDrive/AGCN/Presenter/Baseline-Qwen3-v0.1')
dest.mkdir(parents=True, exist_ok=True)
for name in ['qwen3_4b_current_v0.1.jsonl','qwen3_4b_current_v0.1_summary.json','qwen3_4b_current_v0.1_runtime.json']:
    shutil.copy2(pathlib.Path('training/baselines') / name, dest / name)
print('Salvo em:', dest)


## 7. Baixar os dois resultados pro celular e mandar no chat

In [ ]:
from google.colab import files
files.download('training/baselines/qwen3_4b_current_v0.1_summary.json')
files.download('training/baselines/qwen3_4b_current_v0.1.jsonl')
